# UCI student dropout - centralized, FedAvg, FedProx, and XAI

This notebook reproduces the UCI experiment from the AIED 2025 paper. It downloads the official [Predict Students' Dropout and Academic Success dataset](https://archive.ics.uci.edu/dataset/697/predict+students+dropout+and+academic+success), removes the `Enrolled` class, maps `Graduate=0` and `Dropout=1`, performs a stratified 80/20 split, fits scaling on the training set only, and evaluates the three learning settings.

Set `AIED_PAPER_MODE=1` before launching Jupyter for 10 repetitions, 50 FL rounds, 2 local epochs, and 10 clients. The default is a short pipeline smoke test. Set `AIED_RUN_XAI=1` to execute the final explainability cell.

In [ ]:
import json
import os
import random
import urllib.request
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import display
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

DEFAULT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
REPO_ROOT = Path(os.environ.get("AIED_REPO_ROOT", DEFAULT_ROOT)).resolve()
DATA_DIR = REPO_ROOT / "data" / "uci"
RAW_DIR = DATA_DIR / "raw"
RESULTS_DIR = REPO_ROOT / "results" / "uci"
PAPER_MODE = os.environ.get("AIED_PAPER_MODE", "0") == "1"
RUN_XAI = os.environ.get("AIED_RUN_XAI", "0") == "1"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CONFIG = {
    "repeats": 10 if PAPER_MODE else 1,
    "central_epochs": 100 if PAPER_MODE else 2,
    "clients": 10 if PAPER_MODE else 3,
    "rounds": 50 if PAPER_MODE else 2,
    "local_epochs": 2 if PAPER_MODE else 1,
    "batch_size": 64,
    "lr": 0.02,
    "mu": 0.01,
    "fast_max_rows": None if PAPER_MODE else 1200,
}
print({"paper_mode": PAPER_MODE, "device": str(DEVICE), **CONFIG})

## 1. Download and prepare the data

The archive is downloaded from UCI's official static endpoint. UCI reports 4,424 rows and 36 features under a [CC BY 4.0 license](https://doi.org/10.24432/C5MC89). The paper's binary task retains 2,209 Graduate and 1,421 Dropout rows.

In [ ]:
UCI_DOWNLOAD = "https://archive.ics.uci.edu/static/public/697/predict+students+dropout+and+academic+success.zip"
data_csv = RAW_DIR / "data.csv"
if not data_csv.exists():
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    archive = DATA_DIR / "uci_697.zip"
    print(f"Downloading {UCI_DOWNLOAD}")
    urllib.request.urlretrieve(UCI_DOWNLOAD, archive)
    with zipfile.ZipFile(archive) as zf:
        zf.extractall(RAW_DIR)

data = pd.read_csv(data_csv, sep=";")
data.columns = [column.strip() for column in data.columns]
data = data.loc[data["Target"].isin(["Graduate", "Dropout"])].copy()
data["Target"] = data["Target"].map({"Graduate": 0, "Dropout": 1}).astype("int64")
assert data.shape == (3630, 37), data.shape
assert data["Target"].value_counts().to_dict() == {0: 2209, 1: 1421}
display(data.head())
display(data["Target"].value_counts().rename(index={0: "Graduate", 1: "Dropout"}))

## 2. Shared model and federated algorithms

All methods use the same `36 -> 30 -> 10 -> 2` architecture, Adam at learning rate 0.02, batch size 64, and inverse-frequency weighted cross-entropy. FedProx adds the published coefficient `mu=0.01`.

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True, warn_only=True)


class DropoutMLP(nn.Module):
    """Two-hidden-layer network used in the paper (30 and 10 units)."""
    def __init__(self, input_dim: int):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, 30),
            nn.ReLU(),
            nn.Linear(30, 10),
            nn.ReLU(),
            nn.Linear(10, 2),
        )

    def forward(self, x):
        return self.layers(x)


def inverse_frequency_weights(y, device):
    counts = np.bincount(np.asarray(y, dtype=np.int64), minlength=2)
    if np.any(counts == 0):
        raise ValueError(f"Both classes must be present in the training split; got {counts.tolist()}")
    weights = len(y) / (2.0 * counts)
    return torch.tensor(weights, dtype=torch.float32, device=device)


def make_loader(X, y, batch_size, shuffle, seed):
    dataset = TensorDataset(
        torch.as_tensor(X, dtype=torch.float32),
        torch.as_tensor(y, dtype=torch.long),
    )
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, generator=generator)


def train_central(X, y, epochs, batch_size, lr, seed, device):
    set_seed(seed)
    model = DropoutMLP(X.shape[1]).to(device)
    criterion = nn.CrossEntropyLoss(weight=inverse_frequency_weights(y, device))
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loader = make_loader(X, y, batch_size, True, seed + 1)
    for _ in range(epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
    return model


def partition_clients(X, y, n_clients, seed):
    if n_clients > len(y):
        raise ValueError("The number of clients cannot exceed the number of training rows.")
    rng = np.random.default_rng(seed)
    partitions = np.array_split(rng.permutation(len(y)), n_clients)
    return [(X[idx], y[idx]) for idx in partitions]


def train_federated(X, y, n_clients, rounds, local_epochs, batch_size, lr, mu, seed, device):
    """FedAvg when mu=0, and FedProx when mu>0."""
    set_seed(seed)
    global_model = DropoutMLP(X.shape[1]).to(device)
    clients = partition_clients(X, y, n_clients, seed + 1)
    class_weights = inverse_frequency_weights(y, device)
    client_sizes = np.asarray([len(cy) for _, cy in clients], dtype=np.float64)
    aggregation_weights = client_sizes / client_sizes.sum()

    for round_idx in range(rounds):
        local_states = []
        for client_idx, (client_X, client_y) in enumerate(clients):
            local_model = DropoutMLP(X.shape[1]).to(device)
            local_model.load_state_dict(global_model.state_dict())
            global_reference = [p.detach().clone() for p in global_model.parameters()]
            criterion = nn.CrossEntropyLoss(weight=class_weights)
            optimizer = torch.optim.Adam(local_model.parameters(), lr=lr)
            loader_seed = seed + 10_000 * round_idx + client_idx
            loader = make_loader(client_X, client_y, batch_size, True, loader_seed)

            for _ in range(local_epochs):
                local_model.train()
                for xb, yb in loader:
                    xb, yb = xb.to(device), yb.to(device)
                    optimizer.zero_grad(set_to_none=True)
                    loss = criterion(local_model(xb), yb)
                    if mu > 0:
                        proximal = sum(
                            torch.sum((local - reference) ** 2)
                            for local, reference in zip(local_model.parameters(), global_reference)
                        )
                        loss = loss + (mu / 2.0) * proximal
                    loss.backward()
                    optimizer.step()
            local_states.append({k: v.detach().clone() for k, v in local_model.state_dict().items()})

        aggregated = {}
        for name in global_model.state_dict():
            aggregated[name] = sum(
                float(weight) * state[name]
                for weight, state in zip(aggregation_weights, local_states)
            )
        global_model.load_state_dict(aggregated)
    return global_model


def evaluate(model, X, y, f1_average, device):
    model.eval()
    with torch.no_grad():
        logits = model(torch.as_tensor(X, dtype=torch.float32, device=device))
        probabilities = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
        predictions = logits.argmax(dim=1).cpu().numpy()
    return {
        "accuracy": accuracy_score(y, predictions),
        "f1": f1_score(y, predictions, average=f1_average, zero_division=0),
        "auc": roc_auc_score(y, probabilities),
    }


def run_three_methods(X_train, y_train, X_test, y_test, config, f1_average, seed, device):
    central = train_central(
        X_train, y_train, config["central_epochs"], config["batch_size"],
        config["lr"], seed + 100, device,
    )
    fedavg = train_federated(
        X_train, y_train, config["clients"], config["rounds"],
        config["local_epochs"], config["batch_size"], config["lr"],
        0.0, seed + 200, device,
    )
    fedprox = train_federated(
        X_train, y_train, config["clients"], config["rounds"],
        config["local_epochs"], config["batch_size"], config["lr"],
        config["mu"], seed + 200, device,
    )
    models = {"Central": central, "FedAvg": fedavg, "FedProx": fedprox}
    metrics = {name: evaluate(model, X_test, y_test, f1_average, device) for name, model in models.items()}
    return metrics, models

## 3. Repeated experiment

In [ ]:
working = data
if CONFIG["fast_max_rows"] and len(working) > CONFIG["fast_max_rows"]:
    working, _ = train_test_split(
        working, train_size=CONFIG["fast_max_rows"], stratify=working["Target"], random_state=0
    )

X_all = working.drop(columns="Target").to_numpy(dtype=np.float32)
y_all = working["Target"].to_numpy(dtype=np.int64)
feature_names = working.drop(columns="Target").columns.to_list()
rows = []
last_context = None

for repeat in range(CONFIG["repeats"]):
    X_train, X_test, y_train, y_test = train_test_split(
        X_all, y_all, test_size=0.2, stratify=y_all, random_state=repeat
    )
    scaler = StandardScaler().fit(X_train)
    X_train = scaler.transform(X_train).astype(np.float32)
    X_test = scaler.transform(X_test).astype(np.float32)
    metrics, models = run_three_methods(
        X_train, y_train, X_test, y_test, CONFIG, "macro", repeat, DEVICE
    )
    for method, values in metrics.items():
        rows.append({"repeat": repeat, "method": method, **values})
    last_context = {
        "model": models["FedProx"], "X_train": X_train, "X_test": X_test,
        "feature_names": feature_names,
    }

run_results = pd.DataFrame(rows)
summary = run_results.groupby("method")[["accuracy", "f1", "auc"]].agg(["mean", "std"])
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
run_results.to_csv(RESULTS_DIR / "uci_metrics.csv", index=False)
torch.save(last_context["model"].state_dict(), RESULTS_DIR / "uci_fedprox_last_seed.pt")
display(summary.round(4))

## 4. Compare with the paper

UCI uses macro F1. Small differences are expected because this notebook fixes train/test scaling leakage present in the archived research cells.

In [ ]:
published = pd.DataFrame(
    {
        "accuracy": [0.899, 0.904, 0.914],
        "f1": [0.865, 0.896, 0.906],
        "auc": [0.890, 0.941, 0.951],
    },
    index=["Central", "FedAvg", "FedProx"],
)
display(published.rename_axis("method").style.set_caption("Published UCI results"))

## 5. Local explanations (LIME, Integrated Gradients, Gradient SHAP)

In [ ]:
def explain_with_captum(context, sample_index=10, top_k=15):
    """Plot local LIME, Integrated Gradients, and Gradient SHAP attributions."""
    if not RUN_XAI:
        print("XAI skipped. Set AIED_RUN_XAI=1 before launching Jupyter to run this section.")
        return None
    from captum.attr import GradientShap, IntegratedGradients, Lime

    model = context["model"].to(DEVICE).eval()
    X_train = context["X_train"]
    X_test = context["X_test"]
    feature_names = np.asarray(context["feature_names"])
    sample_index = min(sample_index, len(X_test) - 1)
    sample = torch.as_tensor(X_test[[sample_index]], dtype=torch.float32, device=DEVICE)
    background = torch.as_tensor(X_train[: min(64, len(X_train))], dtype=torch.float32, device=DEVICE)

    ig_values = IntegratedGradients(model).attribute(sample, baselines=torch.zeros_like(sample), target=1)
    gs_values = GradientShap(model).attribute(sample, baselines=background, target=1, n_samples=50)
    lime_values = Lime(model).attribute(
        sample, target=1, n_samples=200, perturbations_per_eval=32,
        baselines=torch.zeros_like(sample),
    )
    values = {
        "LIME": lime_values.detach().cpu().numpy().ravel(),
        "Integrated Gradients": ig_values.detach().cpu().numpy().ravel(),
        "Gradient SHAP": gs_values.detach().cpu().numpy().ravel(),
    }

    fig, axes = plt.subplots(1, 3, figsize=(18, 6), constrained_layout=True)
    for ax, (name, attribution) in zip(axes, values.items()):
        idx = np.argsort(np.abs(attribution))[-top_k:]
        colors = np.where(attribution[idx] >= 0, "#2f6b9a", "#d18f00")
        ax.barh(feature_names[idx], attribution[idx], color=colors)
        ax.axvline(0, color="black", linewidth=0.8)
        ax.set_title(name)
        ax.set_xlabel("Attribution toward dropout/positive class")
    prediction = torch.softmax(model(sample), dim=1)[0, 1].item()
    fig.suptitle(f"Sample {sample_index}: positive-class probability = {prediction:.3f}")
    plt.show()
    return values

In [ ]:
explain_with_captum(last_context, sample_index=10)